# Регресія тривалості впливу ДТП: лінійна модель vs градієнтний бустинг

Дані — результат `regression_data_preparation.ipynb` (`data/processed/regression/`): ознаки вже очищені, закодовані й масштабовані, ціль `Log_Duration = log1p(Duration_Minutes)` (тривалість у хвилинах має довгий правий хвіст; у лог-шкалі розподіл близький до симетричного).

**Два сімейства моделей** (обидва — scikit-learn):
* **Сімейство 1 — `Ridge`**: лінійна регресія з L2-регуляризацією (при `alpha → 0` — звичайний МНК).
* **Сімейство 2 — `HistGradientBoostingRegressor`**: градієнтний бустинг дерев на гістограмах ознак — враховує нелінійності й взаємодії, масштабується на мільйони записів.

**Етапи:**
1. Завантаження та експрес-перевірка даних.
2. Підбір гіперпараметрів кожного сімейства на підвибірці train (`RandomizedSearchCV`, 5-fold CV, RMSE).
3. Навчання на повному train з базовими та підібраними параметрами — 2 варіанти на сімейство; таблиця метрик на train.
4. Valid: метрики всіх варіантів → лідер кожного сімейства за RMSE → графіки train vs valid.
5. Test: фінальна оцінка обох лідерів.
6. SHAP: beeswarm і bar chart важливості ознак для обох лідерів.

Бібліотека — scikit-learn: Spark на Windows без `winutils.exe` не читає parquet, а готові дані — звичайні parquet-файли.

## Етап 1. Завантаження та експрес-перевірка даних

In [ ]:
import os
import json
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "main.py").exists() else Path.cwd().parent
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import pyarrow.parquet as pq
from IPython.display import display

SEED = 42
# Частка train для машин з малою пам'яттю (повний train у float64 — ~8 ГБ); 1 — весь train
TRAIN_FRACTION = float(os.environ.get("TRAIN_FRACTION", "1"))
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "regression"
MODELS_DIR = PROJECT_ROOT / "data" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

meta = json.loads((DATA_DIR / "metadata.json").read_text(encoding="utf-8"))
FEATURES = meta["feature_names"]
TARGETS = ["Log_Duration", "Duration_Minutes"]


def load(split, dtype, fraction=1.0):
    files = sorted((DATA_DIR / split).glob("part-*.parquet"))
    rows = [pq.ParquetFile(f).metadata.num_rows for f in files]
    rng = np.random.default_rng(SEED)
    masks = [rng.random(r) < fraction if fraction < 1 else None for r in rows]
    n = sum(r if m is None else int(m.sum()) for r, m in zip(rows, masks))
    X, targets, pos = np.empty((n, len(FEATURES)), dtype=dtype), [], 0
    for f, m in zip(files, masks):
        part = pd.read_parquet(f, columns=FEATURES + TARGETS)
        if m is not None:
            part = part[m]
        X[pos:pos + len(part)] = part[FEATURES].to_numpy(dtype=dtype)
        targets.append(part[TARGETS])
        pos += len(part)
    return X, pd.concat(targets, ignore_index=True)


t0 = time.time()
X_train, t_train = load("train", np.float64, TRAIN_FRACTION)
X_valid, t_valid = load("valid", np.float32)
X_test, t_test = load("test", np.float32)
SPLITS = {"train": (X_train, t_train), "valid": (X_valid, t_valid), "test": (X_test, t_test)}
y_train, y_valid, y_test = (t["Log_Duration"].to_numpy() for t in (t_train, t_valid, t_test))
print(f"train {X_train.shape}{f' ({TRAIN_FRACTION:.0%} train)' if TRAIN_FRACTION < 1 else ''}, "
      f"valid {X_valid.shape}, test {X_test.shape} — {time.time() - t0:.0f} c")

Матриці ознак зберігаються як numpy-масиви (train — float64, бо обидві моделі рахують у подвійній точності; valid / test — float32), тому для перегляду обгортаємо їх у `DataFrame` без копіювання. При 230 стовпцях `.info()` виводить коротке зведення (типи, пам'ять), а `.describe()` — по рядку на ознаку (показано перші й останні). Пропусків немає — перевіряємо явно. Наприкінці — розподіл цілі на трьох вибірках.

In [ ]:
for split, (X, t) in SPLITS.items():
    df = pd.DataFrame(X, columns=FEATURES, copy=False)
    print(f"\n{'=' * 30} {split}: {X.shape[0]:,} записів × {X.shape[1]} ознак {'=' * 30}")
    display(pd.DataFrame(X[:5], columns=FEATURES).join(t.head(5)))
    df.info()
    print(f"пропусків (NaN): {int(np.isnan(X).sum())}")
    # describe() копіює всі числові стовпці; порціями по 40 стовпців копія train — ~1.5 ГБ замість ~8 ГБ
    display(pd.concat([df.iloc[:, i:i + 40].describe() for i in range(0, X.shape[1], 40)], axis=1).T)

In [ ]:
pd.concat({split: t[TARGETS].describe() for split, (_, t) in SPLITS.items()}, axis=1).round(3)

## Етап 2. Підбір гіперпараметрів

**Чому на підвибірці.** Пошук — це десятки навчань моделі (кандидати × 5 фолдів); на ~4.5 млн записів це багато годин, а 5 копій фолдів train не вмістилися б у пам'ять. Тому підбираємо на **підвибірці 100 000 записів (~2% train)**, стратифікованій за **децилями цілі**: частки коротких, типових і довгих подій такі самі, як у train, а випадковий відбір записів зберігає зв'язки між ознаками.

**Пошук:** `RandomizedSearchCV`, `KFold(5)`, метрика — **RMSE у лог-шкалі** (`Log_Duration`): саме цю величину мінімізують обидві моделі, і за нею ж обираємо лідерів на valid.

* **Ridge:** сила регуляризації `alpha` (логарифмічна шкала 10⁻³…10⁴).
* **HistGradientBoosting:** `learning_rate`, кількість ітерацій `max_iter`, розмір дерев `max_leaf_nodes`, `min_samples_leaf`, `l2_regularization`, частка ознак на розбиття `max_features`. Рання зупинка вимкнена (`early_stopping=False`): на повному train вона відрізала б 10% даних копією матриці (+8 ГБ), тож кількість ітерацій — теж гіперпараметр.

Фолди й кандидати навчаються паралельно (`n_jobs=-1`, по одному потоку на процес): на 80 000 записів багатопотокові обчислення всередині однієї моделі дають більше накладних витрат, ніж користі (логістична регресія з 12 потоками BLAS навчається в ~10 разів довше, ніж з одним).

Застереження: оптимальні значення залежать від обсягу даних — на 4.5 млн записів регуляризація впливає менше, а дерева могли б бути глибшими, тож підібрані параметри радше консервативні.

In [ ]:
from scipy.stats import loguniform, randint, uniform
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, RandomizedSearchCV, train_test_split

TUNE_ROWS = 100_000
deciles = pd.qcut(y_train, 10, labels=False, duplicates="drop")
tune_idx = (np.arange(len(y_train)) if len(y_train) <= TUNE_ROWS else
            train_test_split(np.arange(len(y_train)), train_size=TUNE_ROWS, stratify=deciles, random_state=SEED)[0])
X_tune, y_tune = X_train[tune_idx], y_train[tune_idx]
print(f"Підвибірка для підбору: {len(tune_idx):,} записів ({100 * len(tune_idx) / len(y_train):.1f}% train)")
pd.DataFrame({"train": pd.Series(y_train).describe(), "підвибірка": pd.Series(y_tune).describe()}).T.round(4)

In [ ]:
FAMILIES = {
    "Ridge": {"base": Ridge(copy_X=False), "space": {"alpha": loguniform(1e-3, 1e4)}, "n_iter": 12},
    "HGB": {"base": HistGradientBoostingRegressor(early_stopping=False, random_state=SEED),
            "space": {"learning_rate": loguniform(0.03, 0.3), "max_iter": randint(100, 201),
                      "max_leaf_nodes": randint(15, 128), "min_samples_leaf": randint(20, 501),
                      "l2_regularization": loguniform(1e-3, 10), "max_features": uniform(0.5, 0.5)}, "n_iter": 12},
}
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
best_params, search_tables = {}, {}
for fam, spec in FAMILIES.items():
    t0 = time.time()
    # copy_X=True лише для пошуку: фолди — копії підвибірки, а X_tune використовується повторно
    base = clone(spec["base"]).set_params(copy_X=True) if fam == "Ridge" else clone(spec["base"])
    search = RandomizedSearchCV(base, spec["space"], n_iter=spec["n_iter"], scoring="neg_root_mean_squared_error",
                                cv=cv, refit=False, random_state=SEED, n_jobs=-1)
    search.fit(X_tune, y_tune)
    best_params[fam] = {k: (v.item() if hasattr(v, "item") else v) for k, v in search.best_params_.items()}
    search_tables[fam] = (pd.DataFrame(search.cv_results_["params"])
                          .assign(**{"RMSE log (CV)": -search.cv_results_["mean_test_score"],
                                     "std": search.cv_results_["std_test_score"]})
                          .sort_values("RMSE log (CV)"))
    print(f"{fam}: {time.time() - t0:.0f} c, найкращий RMSE log (CV) {-search.best_score_:.4f}, best_params_ = {best_params[fam]}")

In [ ]:
for fam, table in search_tables.items():
    print(f"{fam}: топ-5 кандидатів з {len(table)}")
    display(table.head(5).round(4))

## Етап 3. Навчання та первинне оцінювання моделей

На повному train навчаємо по 2 варіанти кожного сімейства: **базові параметри** (значення scikit-learn за замовчуванням: `Ridge(alpha=1)`, бустинг — 100 ітерацій без ранньої зупинки) і **підібрані** `best_params_`.

`Ridge` з `copy_X=False` центрує матрицю train на місці (інакше тримав би в пам'яті другу копію на 8 ГБ); після навчання вихідні значення відновлюються й перевіряються.

**Метрики** — у лог-шкалі (в ній моделі навчались) і **в хвилинах** (прогноз = `expm1(прогноз)`): R², MAE, MSE, RMSE, MAPE (тривалість ≥ 1 хв, тож ділення на нуль немає), MedAE (медіанна похибка — типова, нечутлива до хвоста). Прогноз `expm1(ŷ)` оцінює **медіану** тривалості для таких умов, а не середнє, тому MedAE у хвилинах — найчесніша метрика в хвилинах.

In [ ]:
VARIANTS = {f"{fam} {kind}": (fam, clone(spec["base"]).set_params(**(best_params[fam] if kind == "tuned" else {})))
            for fam, spec in FAMILIES.items() for kind in ("base", "tuned")}

check_rows = X_train[:1_000].copy()
models, fit_info = {}, {}
for name, (fam, est) in VARIANTS.items():
    t0 = time.time()
    if fam == "Ridge":
        X_mean = X_train.mean(axis=0)
        est.fit(X_train, y_train)
        X_train += X_mean
        assert np.allclose(X_train[:1_000], check_rows, atol=1e-9), "train не відновлено після центрування"
    else:
        est.fit(X_train, y_train)
    models[name] = est
    fit_info[name] = {"секунд": round(time.time() - t0)}
    print(f"{name}: {fit_info[name]}")

In [ ]:
from sklearn.metrics import (r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error,
                             mean_absolute_percentage_error, median_absolute_error)


def metrics(t, pred):
    out = {}
    for scale, y, p in [("log", t["Log_Duration"].to_numpy(), pred), ("хв", t["Duration_Minutes"].to_numpy(), np.expm1(pred))]:
        out.update({f"R² ({scale})": r2_score(y, p), f"MAE ({scale})": mean_absolute_error(y, p),
                    f"MSE ({scale})": mean_squared_error(y, p), f"RMSE ({scale})": root_mean_squared_error(y, p),
                    f"MAPE, % ({scale})": 100 * mean_absolute_percentage_error(y, p),
                    f"MedAE ({scale})": median_absolute_error(y, p)})
    return out


def metrics_table(X, t):
    table = pd.DataFrame({(VARIANTS[name][0], name): metrics(t, m.predict(X)) for name, m in models.items()}).T
    return table.rename_axis(["сімейство", "модель"])


results_train = metrics_table(X_train, t_train)
results_train.round(4)

## Етап 4. Оцінювання на валідації

Valid не бере участі в навчанні, тож на ньому порівнюємо варіанти й обираємо **лідера кожного сімейства за RMSE у лог-шкалі** (та сама метрика, що й під час підбору).

In [ ]:
results_valid = metrics_table(X_valid, t_valid)
LEADERS = {fam: results_valid.loc[fam, "RMSE (log)"].idxmin() for fam in FAMILIES}
print("Лідери за RMSE (log) на valid:", LEADERS)
results_valid.round(4)

**Графіки train vs valid** для обох лідерів (у лог-шкалі; для читабельності — 200 000 випадкових записів кожної вибірки, щільність точок — колір):
* **Actual vs Predicted** — ідеальний прогноз лежить на діагоналі;
* **Residual plot** — залишок $y - \hat y$ проти прогнозу: без тренду й з однаковим розкидом, якщо модель не має систематичних помилок.

Схожі картини на train і valid означають, що перенавчання немає.

In [ ]:
def sample(n, size=200_000):
    return np.random.default_rng(SEED).choice(n, size=min(size, n), replace=False)


def plot_actual_vs_pred(ax, y, pred, title):
    ax.hexbin(y, pred, gridsize=70, bins="log", cmap="viridis", mincnt=1)
    lims = [min(y.min(), pred.min()), max(y.max(), pred.max())]
    ax.plot(lims, lims, "r--", lw=1)
    ax.set(title=title, xlabel="Фактичне log1p(тривалість)", ylabel="Прогноз")


def plot_residuals(ax, y, pred, title):
    ax.hexbin(pred, y - pred, gridsize=70, bins="log", cmap="magma", mincnt=1)
    ax.axhline(0, color="black", lw=1)
    ax.set(title=title, xlabel="Прогноз", ylabel="Залишок y − ŷ")


preds = {}
for fam, name in LEADERS.items():
    for split, X, y in [("train", X_train, y_train), ("valid", X_valid, y_valid)]:
        idx = sample(len(y))
        preds[(fam, split)] = (y[idx], models[name].predict(X[idx]))

for plot, label in [(plot_actual_vs_pred, "Actual vs Predicted"), (plot_residuals, "Residual plot")]:
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    for row, (fam, name) in enumerate(LEADERS.items()):
        for col, split in enumerate(["train", "valid"]):
            plot(axes[row, col], *preds[(fam, split)], f"{name} — {split}")
    fig.suptitle(f"{label}: лідери, train vs valid")
    plt.tight_layout()
    plt.show()

## Етап 5. Фінальне оцінювання на тестовій вибірці

Test використовуємо один раз — для двох лідерів, обраних на valid. Це чесна оцінка якості на нових даних.

In [ ]:
pred_test = {fam: models[name].predict(X_test) for fam, name in LEADERS.items()}
results_test = pd.DataFrame({(fam, LEADERS[fam]): metrics(t_test, p) for fam, p in pred_test.items()}).T
results_test.rename_axis(["сімейство", "лідер (test)"]).round(4)

In [ ]:
idx = sample(len(y_test))
fig, axes = plt.subplots(2, 3, figsize=(21, 12))
for row, (fam, name) in enumerate(LEADERS.items()):
    y, p = y_test[idx], pred_test[fam][idx]
    plot_actual_vs_pred(axes[row, 0], y, p, f"{name} — test: Actual vs Predicted")
    plot_residuals(axes[row, 1], y, p, f"{name} — test: залишки vs прогноз")
    resid = y_test - pred_test[fam]
    sns.histplot(resid, bins=120, ax=axes[row, 2], color="slateblue")
    axes[row, 2].axvline(0, color="black", lw=1)
    axes[row, 2].set(title=f"{name} — test: розподіл залишків (середнє {resid.mean():+.3f}, std {resid.std():.3f})",
                     xlabel="Залишок y − ŷ (log)")
plt.tight_layout()
plt.show()

In [ ]:
minutes = t_test["Duration_Minutes"].to_numpy()
bins = pd.cut(minutes, [0, 30, 60, 120, 240, 480, np.inf], labels=["≤30", "30–60", "60–120", "120–240", "240–480", ">480"])
pd.DataFrame({"діапазон, хв": bins, "фактична": minutes,
              **{f"прогноз {fam}": np.expm1(p) for fam, p in pred_test.items()}}).groupby("діапазон, хв", observed=True).agg(
    записів=("фактична", "size"), медіана_факт=("фактична", "median"),
    **{f"медіана_прогноз_{fam}": (f"прогноз {fam}", "median") for fam in pred_test},
).round(1)

In [ ]:
pd.DataFrame({(fam, split): table.loc[(fam, LEADERS[fam]), ["R² (log)", "RMSE (log)", "MAE (хв)", "MedAE (хв)", "MAPE, % (хв)"]]
              for fam in FAMILIES
              for split, table in [("train", results_train), ("valid", results_valid), ("test", results_test)]}).T.round(4)

## Етап 6. SHAP-аналіз та інтерпретованість

**SHAP** розкладає кожен прогноз на внески ознак: прогноз (у лог-шкалі) = базове значення (середній прогноз) + сума внесків. Внесок +0.1 ≈ тривалість ×1.105.
* **Ridge:** `shap.LinearExplainer` (маскер `Independent` — середні з 2 000 записів train): внесок ознаки $j$ — рівно $\beta_j\,(x_j - \bar x_j)$.
* **HistGradientBoosting:** `shap.TreeExplainer` — точні SHAP-значення для дерев (з урахуванням взаємодій ознак).

**Вибірка:** 5 000 записів test, **стратифікованих за децилями цілі** — короткі й довгі події представлені в тих самих частках, що й у test; рахувати SHAP на всіх 1.3 млн записів немає потреби.

Для кожного лідера: **bar chart** (середній |SHAP|, топ-20 стовпців матриці ознак), **beeswarm** (кожна точка — запис, положення — внесок, колір — значення ознаки) і таблиця важливості **вихідних ознак** (внески one-hot категорій і бінів однієї ознаки додаються).

In [ ]:
import shap

rng = np.random.default_rng(SEED)
background = X_train[rng.choice(len(X_train), size=2_000, replace=False)]
explain_idx, _ = train_test_split(np.arange(len(y_test)), train_size=5_000,
                                  stratify=pd.qcut(y_test, 10, labels=False, duplicates="drop"), random_state=SEED)
X_explain = X_test[explain_idx].astype(np.float64)


def feature_group(name):
    if name.startswith("DoW_"):
        return "DayOfWeek"
    for sep in ("=", " in ["):
        if sep in name:
            return name.split(sep)[0]
    return name


GROUPS = [feature_group(f) for f in FEATURES]
shap_values = {}
for fam, name in LEADERS.items():
    t0 = time.time()
    model = models[name]
    explainer = (shap.LinearExplainer(model, shap.maskers.Independent(background, max_samples=len(background)))
                 if fam == "Ridge" else shap.TreeExplainer(model))
    sv = explainer(X_explain)
    sv.feature_names = FEATURES
    assert np.allclose(sv.base_values + sv.values.sum(axis=1), model.predict(X_explain), atol=1e-4), "база + внески ≠ прогноз"
    shap_values[fam] = sv
    print(f"{name}: SHAP за {time.time() - t0:.0f} c")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(22, 8))
for ax, (fam, sv) in zip(axes, shap_values.items()):
    imp = pd.Series(np.abs(sv.values).mean(axis=0), index=FEATURES).nlargest(20)
    imp[::-1].plot.barh(ax=ax, color="steelblue")
    ax.set_title(f"{LEADERS[fam]}: середній |SHAP|, топ-20 ознак")
    ax.set_xlabel("середній |внесок| у log1p(тривалість)")
    ax.tick_params(axis="y", labelsize=8)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(24, 9))
for ax, (fam, sv) in zip(axes, shap_values.items()):
    shap.plots.beeswarm(sv, max_display=15, ax=ax, show=False, plot_size=None)
    ax.set_title(f"{LEADERS[fam]}: внески в log1p(тривалість) (колір — значення ознаки)")
    ax.tick_params(axis="y", labelsize=8)
plt.tight_layout()
plt.show()

In [ ]:
grouped = pd.DataFrame({LEADERS[fam]: pd.DataFrame(sv.values, columns=FEATURES).T.groupby(GROUPS).sum().T.abs().mean()
                        for fam, sv in shap_values.items()})
grouped.assign(**{f"ранг: {c}": grouped[c].rank(ascending=False).astype(int) for c in grouped.columns}) \
    .sort_values(grouped.columns[0], ascending=False).head(20).round(4)

## Збереження лідерів

In [ ]:
for fam, name in LEADERS.items():
    path = MODELS_DIR / f"regression_leader_{fam.lower()}.joblib"
    joblib.dump({"family": fam, "variant": name, "model": models[name], "params": models[name].get_params(),
                 "features": FEATURES, "target": "Log_Duration", "inverse": "expm1"}, path)
    print("Збережено:", path.name)

## Висновки

_(заповнюється після запуску)_

**Нові дані: невідомий рік джерела.** `Source_Year` прив'язана до років train. Для запису з новим роком (наприклад, `Source1_2026`) **замінюємо `Source_Year` на останній відомий рік того ж джерела** (`Source1_2023`, `Source2_2022`, `Source3_2022`; див. `metadata.json` → `unseen_source_year`) — інакше запис або відкинеться, або буде сприйнятий як референсна категорія `Source3_2016` з іншим режимом фіксації часу завершення. Для класифікації перевірка «на майбутньому» підтвердила, що з цим правилом якість на новому році не падає (accuracy 0.969 проти 0.744 без нього); правило коректне, поки джерело не змінює спосіб фіксації даних.